# Notebook Python confusion generation

In [1]:
import pandas as pd
import json

In [2]:
df = pd.read_csv("Metagenomic_gold_standard_consensus.tsv", sep="\t", encoding="utf-8")

print("Columns detected in file:")
for col in df.columns:
    print(f"[{col}]")

Columns detected in file:
[Source]
[Tool name]
[Topic]
[Operation]
[Input data]
[Output data]
[Input format]
[Output format]


In [9]:
def process_annotation_field(field_value):
    """
    Process annotation fields by:
    - converting to lowercase
    - removing excess whitespace
    - splitting on commas
    - returning a clean list of annotations
    """
    if pd.isna(field_value) or field_value == "" or field_value is None:
        return []

    # Convert to lowercase
    text = str(field_value).lower()

    # Normalize excessive whitespace
    text = " ".join(text.split())

    # Split by commas and strip whitespace
    annotations = [item.strip() for item in text.split(",")]

    # Remove empty strings
    annotations = [item for item in annotations if item]

    return annotations


def load_tsv_and_create_json(
    tsv_file="Metagenomic_gold_standard_consensus.tsv",
    json_file="metagenomic_tools_annotations.json",
):
    """
    Load TSV and create JSON including all sources (consensus_expert_LLM, proposition DeepSeek-V3.1, consensus_expert)
    """
    df = pd.read_csv(tsv_file, sep="\t", encoding="utf-8")
    df.columns = df.columns.str.strip()  # normalize column names

    all_tools = {}
    counter = 1

    for _, row in df.iterrows():
        tool_key = f"tools#{counter}"
        source = row.get("Source", "").strip()

        # Parse all annotation columns robustly
        tool_data = {
            "name": row.get("Tool name", "").strip(),
            "discipline": "Metagenomics",
            "source": source,
            "topic": process_annotation_field(row.get("Topic", "")),
            "operation": process_annotation_field(row.get("Operation", "")),
            "input data": process_annotation_field(row.get("Input data", "")),
            "output data": process_annotation_field(row.get("Output data", "")),
            "input format": process_annotation_field(row.get("Input format", "")),
            "output format": process_annotation_field(row.get("Output format", "")),
        }

        all_tools[tool_key] = tool_data
        counter += 1

    # Save JSON
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(all_tools, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(all_tools)} tools to {json_file}")


In [10]:
def load_json(json_file="metagenomic_tools_annotations.json"):
    try:
        with open(json_file, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return None


In [14]:
def calculate_metrics_per_tool_annotation(json_data):
    """
    For each tool and each annotation type, calculate TP, FP, FN, TN, precision, recall, F1
    using lowercase cleaned annotations from process_annotation_field().
    """

    annotation_fields = [
        "topic",
        "operation",
        "input data",
        "output data",
        "input format",
        "output format",
    ]

    results = []

    # collect all unique tool names (already lowercase)
    tool_names = set(entry["name"] for entry in json_data.values())

    # expected provenances (lowercase, normalized)
    expected_prov = {
        "consensus expert llm",
        "proposition deepseek-v3.1",
        "consensus expert",
    }

    for tool in tool_names:
        # initialize empty sets per provenance
        prov_sets = {
            "consensus expert llm": {field: set() for field in annotation_fields},
            "proposition deepseek-v3.1": {field: set() for field in annotation_fields},
            "consensus expert": {field: set() for field in annotation_fields},
        }

        # fill annotation sets
        for entry in json_data.values():
            if entry["name"] != tool:
                continue

            prov = entry["source"].strip().lower()

            if prov not in prov_sets:
                continue

            for field in annotation_fields:
                prov_sets[prov][field].update(entry.get(field, []))

        # compute confusion metrics for each annotation type
        for field in annotation_fields:
            final_consensus_set = prov_sets["consensus expert llm"][field]
            deepseek_set = prov_sets["proposition deepseek-v3.1"][field]
            expert_set = prov_sets["consensus expert"][field]

            # missing by deepseek
            missing_in_deepseek = final_consensus_set - deepseek_set

            # FN = elements in LLM not predicted by DeepSeek but present in Expert
            FN_set = {x for x in missing_in_deepseek if x in expert_set}

            # mixed contributions = predicted by both even if not matched DeepSeek
            mixed_tp = {x for x in missing_in_deepseek if x not in expert_set}

            # TP: direct intersection + mixed tp
            TP_set = (final_consensus_set & deepseek_set) | mixed_tp

            # FP: predicted by deepseek but not in LLM
            FP_set = deepseek_set - final_consensus_set

            TN_set = set()  # always empty

            TP = len(TP_set)
            FP = len(FP_set)
            FN = len(FN_set)
            TN = len(TN_set)  # always 0

            precision = round(TP / (TP + FP) if (TP + FP) else 0, 2)
            recall = round(TP / (TP + FN) if (TP + FN) else 0, 2)
            f1_score = round(
                (
                    2 * precision * recall / (precision + recall)
                    if (precision + recall)
                    else 0
                ),
                2,
            )

            results.append({
                "tool_name": tool,
                "annotation_type": field,
                "TP": TP,
                "FP": FP,
                "FN": FN,
                "TN": TN,
                "precision": precision,
                "recall": recall,
                "f1_score": f1_score,
                "TP_annotations": sorted(TP_set),
                "FP_annotations": sorted(FP_set),
                "FN_annotations": sorted(FN_set),
            })

    return results


In [15]:
def save_metrics_to_tsv(metrics, tsv_file="confusion_metrics_per_tool_annotation.tsv"):
    df = pd.DataFrame(metrics)
    df.to_csv(tsv_file, sep="\t", index=False)
    print(f"Saved confusion metrics per tool and annotation type to {tsv_file}")


In [16]:
def main():
    # 1️⃣ Regenerate JSON including consensus_expert
    load_tsv_and_create_json(
        tsv_file="Metagenomic_gold_standard_consensus.tsv",
        json_file="metagenomic_tools_annotations.json",
    )

    # 2️⃣ Load JSON
    data = load_json()
    if not data:
        return

    # 3️⃣ Calculate metrics per tool and annotation type
    metrics = calculate_metrics_per_tool_annotation(data)

    # 4️⃣ Save to TSV
    save_metrics_to_tsv(metrics)


if __name__ == "__main__":
    main()


Saved 28 tools to metagenomic_tools_annotations.json
Saved confusion metrics per tool and annotation type to confusion_metrics_per_tool_annotation.tsv


In [20]:
# load the results with pandas
df = pd.read_csv("confusion_metrics_per_tool_annotation.tsv", sep="\t")

# compute the mean F1 score across all tools and annotation types
mean_f1 = df["f1_score"].mean()
print(f"Mean F1 score across all tools and annotation types: {mean_f1:.2f}")
mean_precision = df["precision"].mean()
print(f"Mean Precision across all tools and annotation types: {mean_precision:.2f}")
mean_recall = df["recall"].mean()
print(f"Mean Recall across all tools and annotation types: {mean_recall:.2f}")

print("\n")
# compute mean F1 score for each annotation type
for annotation_type in df["annotation_type"].unique():
    subset = df[df["annotation_type"] == annotation_type]
    mean_f1_type = subset["f1_score"].mean()
    print(f"Mean F1 score for annotation type '{annotation_type}': {mean_f1_type:.2f}")
    mean_precision_type = subset["precision"].mean()
    print(
        f"Mean Precision for annotation type '{annotation_type}': {mean_precision_type:.2f}"
    )
    mean_recall_type = subset["recall"].mean()
    print(
        f"Mean Recall for annotation type '{annotation_type}': {mean_recall_type:.2f}"
    )
    print("\n")

Mean F1 score across all tools and annotation types: 0.43
Mean Precision across all tools and annotation types: 0.42
Mean Recall across all tools and annotation types: 0.50


Mean F1 score for annotation type 'topic': 0.35
Mean Precision for annotation type 'topic': 0.35
Mean Recall for annotation type 'topic': 0.35


Mean F1 score for annotation type 'operation': 0.25
Mean Precision for annotation type 'operation': 0.20
Mean Recall for annotation type 'operation': 0.33


Mean F1 score for annotation type 'input data': 0.35
Mean Precision for annotation type 'input data': 0.35
Mean Recall for annotation type 'input data': 0.36


Mean F1 score for annotation type 'output data': 0.20
Mean Precision for annotation type 'output data': 0.17
Mean Recall for annotation type 'output data': 0.26


Mean F1 score for annotation type 'input format': 0.77
Mean Precision for annotation type 'input format': 0.75
Mean Recall for annotation type 'input format': 0.91


Mean F1 score for annotation type 

# Mapping with EDAM concepts 

In [ ]:
# from the consensus json
json_file = "metagenomic_tools_annotations.json"
data = {}
with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# enumerate all annotation fields
annotation_fields = [
    "Topic",
    "Operation",
    "Input data",
    "Output data",
    "Input format",
    "Output format",
]

all_annotations = set()

for tool_key, tool_data in data.items():
    if tool_data["source"] == "Consensus expert LLM":
        topics = tool_data.get("topic", [])
        operations = tool_data.get("operation", [])
        input_data = tool_data.get("input data", [])
        output_data = tool_data.get("output data", [])
        input_format = tool_data.get("input format", [])
        output_format = tool_data.get("output format", [])
        all_annotations.update(set(topics))
        all_annotations.update(set(operations))
        all_annotations.update(set(input_data))
        all_annotations.update(set(output_data))
        all_annotations.update(set(input_format))
        all_annotations.update(set(output_format))

all_annotations = sorted(all_annotations)
print(f"Total unique annotations found: {len(all_annotations)}")
print("All unique annotations across all tools and fields:")
for annotation in all_annotations:
    print(annotation)

Total unique annotations found: 69
All unique annotations across all tools and fields:
abundance and composition integration
abundance profiling
abundance table
activity profiling
annotated contigs
annotation edition
assembly
bam
binning
binning refinement
bins
bruijn graph
co-abundance
comparative genomics
contigs
coverage profiles
de novo assembly
fasta
fastq
functional annotation
functional annotations
functional genomics
genome
genome annotation
genome assemby
genome completness
genome contamination
genome quality
genome reconstruction
genomic refining
html
image
interactive edition
interactive visualization
jpeg
json
metabolic profiles
metagenome
metagenome assembled genomes mags
metagenomic assembly
metagenomics
newick
phylogenetic inference
phylogenetic tree
phylogenomics
phylogeny
png
quality annotation
raw sequence
sequence abundance profile
sequence analysis
sequence assembly
sequence classification
sequencing
sequencing reads
short reads
svg
taxonomic annotation
taxonomic cl